# Time-Series Aggregation and Rolling Analysis

In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 500

df = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "customer_id": np.random.randint(101, 151, n),
    "order_date": pd.to_datetime(
        np.random.choice(
            pd.date_range("2026-01-01", "2026-03-31"),
            n
        )
    ),
    "sales": np.random.randint(200, 5001, n),
    "status": np.random.choice(
        ["Completed", "Cancelled", "Pending"],
        n,
        p=[0.70, 0.20, 0.10]
    ),
    "category": np.random.choice(
        ["Electronics", "Clothing", "Home", "Books"],
        n
    )
})

df = df.sort_values("order_date").reset_index(drop=True)

df.head()

,order_id,customer_id,order_date,sales,status,category
0,1387,132,2026-01-01,2619,Completed,Electronics
1,1196,137,2026-01-01,485,Completed,Home
2,1365,137,2026-01-01,2621,Completed,Books
3,1165,136,2026-01-01,2190,Cancelled,Electronics
4,1490,128,2026-01-01,2576,Completed,Clothing


#### Q1. Daily sales aggregation
Create a daily report containing:
- Total completed sales per day.
- Number of completed orders per day.
- Average order value per day.
Use groupby() with the order date normalized to the day.

In [9]:
report=df.groupby('order_date').agg(
    Completed_sales=('sales',lambda x: x[df.loc[x.index, 'status'] == 'Completed'].sum()),
    completed_orders=('status',lambda x: (x == 'Completed').sum() ),
    avg_order_val=('sales','mean')
)
report

,Completed_sales,completed_orders,avg_order_val
order_date,,,
2026-01-01,12139,6,2170.125000
2026-01-02,9336,3,3112.000000
2026-01-03,16814,6,3017.857143
2026-01-04,12378,4,2867.125000
2026-01-05,23556,7,3376.333333
...,...,...,...
2026-03-27,15369,5,3263.428571
2026-03-28,4771,3,1776.750000
2026-03-29,4478,1,2165.000000


#### Q2. Daily resampling
Convert order_date into the DataFrame index and use resample('D') to calculate daily completed sales.

Include dates with no completed orders, representing their sales as zero.

In [14]:
daily_completed_sales = (
    df.set_index('order_date')
      .loc[lambda x: x['status'] == 'Completed', 'sales']
      .resample('D')
      .sum()
      .fillna(0)
)

display(daily_completed_sales)

order_date
2026-01-01    12139
2026-01-02     9336
2026-01-03    16814
2026-01-04    12378
2026-01-05    23556
              ...  
2026-03-27    15369
2026-03-28     4771
2026-03-29     4478
2026-03-30    10684
2026-03-31    23941
Freq: D, Name: sales, Length: 90, dtype: int32

#### Q3. Weekly performance
Generate a weekly report containing:
- Total completed sales.
- Number of all orders.
- Number of cancelled orders.
- Cancellation rate as a percentage of all orders.
Use weekly resampling. Make sure the cancellation rate is calculated from weekly counts, not by averaging daily percentages.

In [19]:
df['completed_sales'] = df['sales'].where(
    df['status'].eq('Completed'), 0
)

df['cancelled_order'] = df['status'].eq('Cancelled').astype(int)

weekly_report = (
    df.set_index('order_date')
      .resample('W')
      .agg(
          total_completed_sales=('completed_sales', 'sum'),
          total_orders=('status', 'count'),
          cancelled_orders=('cancelled_order', 'sum')
      )
)

weekly_report['cancellation_rate_pct'] = (
    weekly_report['cancelled_orders']
    / weekly_report['total_orders'].replace(0, float('nan'))
    * 100
).fillna(0)

weekly_report

,total_completed_sales,total_orders,cancelled_orders,cancellation_rate_pct
order_date,,,,
2026-01-04,50667,26,4,15.384615
2026-01-11,93727,40,5,12.500000
2026-01-18,81162,46,10,21.739130
2026-01-25,71404,40,10,25.000000
2026-02-01,65906,35,5,14.285714
2026-02-08,89115,48,10,20.833333
2026-02-15,51682,28,5,17.857143
2026-02-22,75642,41,7,17.073171
2026-03-01,65442,35,6,17.142857


#### Q4. Monthly performance
Create a monthly report for January, February, and March 2026 containing:
- Completed revenue.
- Completed order count.
- Average order value.
- Total order count, including all statuses.
- Cancelled order count.

In [22]:
df['completed_sales'] = df['sales'].where(
    df['status'].eq('Completed'), 0
)

df['completed_order'] = df['status'].eq('Completed').astype(int)

df['cancelled_order'] = df['status'].eq('Cancelled').astype(int)

monthly_report = (
    df.set_index('order_date')
      .loc['2026-01-01':'2026-03-31']
      .resample('MS')
      .agg(
          completed_revenue=('completed_sales', 'sum'),
          completed_order_count=('completed_order', 'sum'),
          total_order_count=('status', 'count'),
          cancelled_order_count=('cancelled_order', 'sum')
      )
)

monthly_report['average_order_value'] = (
    monthly_report['completed_revenue']
    / monthly_report['completed_order_count'].replace(0, float('nan'))
)

monthly_report

,completed_revenue,completed_order_count,total_order_count,cancelled_order_count,average_order_value
order_date,,,,,
2026-01-01,361274,136,183,32,2656.426471
2026-02-01,282441,108,155,30,2615.194444
2026-03-01,315936,115,162,33,2747.269565


#### Q5. groupby() versus resample()
Calculate monthly completed sales in two ways:
1. Using groupby() on the month extracted from order_date.
2. Using resample('ME') on a datetime index.
Compare the results and explain one practical difference between the two approaches.

In [25]:
completed = df[df['status'].eq('Completed')].copy()

monthly_groupby = (
    completed.groupby(completed['order_date'].dt.to_period('M'))['sales']
    .sum()
    .rename('completed_sales')
)

monthly_groupby

order_date
2026-01    361274
2026-02    282441
2026-03    315936
Freq: M, Name: completed_sales, dtype: int32

#### Q6. Seven-day rolling average
Calculate daily completed sales and then create a 7-day rolling average.

Requirements:
- Sort dates chronologically.
- Include days with zero sales.
- Use a seven-day window.
- Compare the first few rolling values with the later values.

In [28]:
daily_sales = (
    df.set_index('order_date')
      .sort_index()
      .assign(
          completed_sales=lambda x: x['sales'].where(
              x['status'].eq('Completed'), 0
          )
      )['completed_sales']
      .resample('D')
      .sum()
)

daily_sales

order_date
2026-01-01    12139
2026-01-02     9336
2026-01-03    16814
2026-01-04    12378
2026-01-05    23556
              ...  
2026-03-27    15369
2026-03-28     4771
2026-03-29     4478
2026-03-30    10684
2026-03-31    23941
Freq: D, Name: completed_sales, Length: 90, dtype: int32

#### Q8. Cumulative sales
Create a daily sales series and calculate cumulative completed sales using cumsum().

Your output should show how total revenue accumulates over the quarter.

In [31]:
daily_sales = (
    df.set_index('order_date')
      .sort_index()
      .loc['2026-01-01':'2026-03-31']
      .assign(
          completed_sales=lambda x: x['sales'].where(
              x['status'].eq('Completed'), 0
          )
      )['completed_sales']
      .resample('D')
      .sum()
)

cumulative_sales = daily_sales.cumsum()

sales_report = pd.DataFrame({
    'daily_completed_sales': daily_sales,
    'cumulative_completed_sales': cumulative_sales
})

sales_report

,daily_completed_sales,cumulative_completed_sales
order_date,,
2026-01-01,12139,12139
2026-01-02,9336,21475
2026-01-03,16814,38289
2026-01-04,12378,50667
2026-01-05,23556,74223
...,...,...
2026-03-27,15369,915777
2026-03-28,4771,920548
2026-03-29,4478,925026
